In [ ]:
# -*- coding: utf-8 -*-
"""
EDA + preprocessing + baseline quality evaluation
for Raman spectra classification: control / endo / exo

Ожидаемая структура:
root/
    control/
        mk1/
            cortex_control_1.txt
            striatum_control_2.txt
        mk2a/
            ...
    endo/
        mk3/
            ...
    exo/
        mk4/
            ...

Формат txt:
#X    #Y    #Wave    #Intensity
...
"""

from __future__ import annotations

import os
import re
import json
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from scipy.sparse.linalg import spsolve

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# Конфиг
# =========================================================

ROOT_DIR = Path(r"E:\Nuclear_Hack_2026\data")   # <-- замените
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    "control": 0,
    "endo": 1,
    "exo": 2,
}
INV_CLASS_MAP = {v: k for k, v in CLASS_MAP.items()}


# =========================================================
# Утилиты
# =========================================================

def seed_everything(seed: int = 42):
    np.random.seed(seed)

seed_everything(42)


@dataclass
class FileMeta:
    filepath: str
    label_name: str
    label: int
    mouse_id: str
    sample_id: str
    region: str


def infer_region(filename: str) -> str:
    name = filename.lower()
    if "cortex" in name:
        return "cortex"
    if "striatum" in name:
        return "striatum"
    return "unknown"


def discover_files(root_dir: Path) -> List[FileMeta]:
    """
    Рекурсивно находит .txt файлы внутри control/endo/exo.
    Предполагается, что mouse_id = имя директории уровня mk*.
    """
    metas = []

    for class_name in ["control", "endo", "exo"]:
        class_dir = root_dir / class_name
        if not class_dir.exists():
            print(f"[WARN] Нет папки: {class_dir}")
            continue

        for txt_path in class_dir.rglob("*.txt"):
            parts = txt_path.parts

            # mouse_id: берём ближайшую родительскую папку
            mouse_id = txt_path.parent.name

            sample_id = txt_path.stem
            region = infer_region(txt_path.name)

            metas.append(
                FileMeta(
                    filepath=str(txt_path),
                    label_name=class_name,
                    label=CLASS_MAP[class_name],
                    mouse_id=mouse_id,
                    sample_id=sample_id,
                    region=region,
                )
            )

    return metas


def robust_read_txt(filepath: str) -> pd.DataFrame:
    """
    Читает Raman txt с колонками #X, #Y, #Wave, #Intensity.
    Поддерживает tab/space-separated формат.
    """
    try:
        df = pd.read_csv(filepath, sep=r"\s+|\t+", engine="python", comment=None)
    except Exception:
        df = pd.read_csv(filepath, sep="\t", engine="python", comment=None)

    # нормализация имен колонок
    rename_map = {}
    for c in df.columns:
        cc = c.strip().replace('"', "")
        rename_map[c] = cc
    df = df.rename(columns=rename_map)

    required = ["#X", "#Y", "#Wave", "#Intensity"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{filepath}: отсутствуют колонки {missing}. Найдены: {df.columns.tolist()}")

    return df[required].copy()


def long_to_cube(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Преобразует long-format (#X, #Y, #Wave, #Intensity) в:
    x_unique, y_unique, wave, cube shape = (n_y, n_x, n_wave)

    Предполагает одинаковую сетку wave для всех (x,y).
    """
    x_unique = np.sort(df["#X"].unique())
    y_unique = np.sort(df["#Y"].unique())
    wave = np.sort(df["#Wave"].unique())

    n_x = len(x_unique)
    n_y = len(y_unique)
    n_wave = len(wave)

    # pivot -> multi-index table
    pivot = df.pivot_table(
        index=["#Y", "#X"],
        columns="#Wave",
        values="#Intensity",
        aggfunc="first"
    )

    # reindex на полную сетку
    full_index = pd.MultiIndex.from_product([y_unique, x_unique], names=["#Y", "#X"])
    pivot = pivot.reindex(full_index)
    pivot = pivot.reindex(columns=wave)

    cube = pivot.values.reshape(n_y, n_x, n_wave)

    return x_unique, y_unique, wave, cube


def flatten_cube(cube: np.ndarray) -> np.ndarray:
    """
    (n_y, n_x, n_wave) -> (n_pixels, n_wave)
    """
    return cube.reshape(-1, cube.shape[-1])


def aggregate_spectra(X: np.ndarray, method: str = "median") -> np.ndarray:
    """
    Агрегация спектров внутри файла.
    """
    if method == "median":
        return np.nanmedian(X, axis=0)
    elif method == "mean":
        return np.nanmean(X, axis=0)
    else:
        raise ValueError(f"Неизвестный method={method}")


# =========================================================
# Предобработка спектров
# =========================================================

def crop_spectra(wave: np.ndarray, X: np.ndarray, wmin: float = 700, wmax: float = 1800):
    mask = (wave >= wmin) & (wave <= wmax)
    return wave[mask], X[:, mask]


def modified_zscore(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + 1e-12
    return 0.6745 * (x - med) / mad


def despike_spectrum(y: np.ndarray, threshold: float = 5.0, window: int = 3) -> np.ndarray:
    """
    Простая реализация spike removal через modified z-score
    на первой производной.
    """
    y = y.copy()
    dy = np.diff(y, prepend=y[0])
    z = np.abs(modified_zscore(dy))
    spike_idx = np.where(z > threshold)[0]

    if len(spike_idx) == 0:
        return y

    for idx in spike_idx:
        left = max(0, idx - window)
        right = min(len(y), idx + window + 1)

        neighbors = np.concatenate([y[left:idx], y[idx+1:right]])
        if len(neighbors) > 0:
            y[idx] = np.median(neighbors)

    return y


def despike_batch(X: np.ndarray, threshold: float = 5.0, window: int = 3) -> np.ndarray:
    return np.vstack([despike_spectrum(x, threshold=threshold, window=window) for x in X])


def smooth_batch(X: np.ndarray, window_length: int = 9, polyorder: int = 3) -> np.ndarray:
    wl = min(window_length, X.shape[1] - 1 if X.shape[1] % 2 == 0 else X.shape[1])
    if wl % 2 == 0:
        wl -= 1
    wl = max(wl, polyorder + 2 + ((polyorder + 2) % 2 == 0))
    if wl % 2 == 0:
        wl += 1
    wl = min(wl, X.shape[1] - 1 if X.shape[1] % 2 == 0 else X.shape[1])

    if wl <= polyorder:
        return X.copy()

    return savgol_filter(X, window_length=wl, polyorder=polyorder, axis=1)


def baseline_als(y: np.ndarray, lam: float = 1e5, p: float = 0.01, niter: int = 10) -> np.ndarray:
    """
    Asymmetric Least Squares baseline correction.
    Возвращает baseline.
    """
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L))
    w = np.ones(L)

    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        Z = W + lam * D.T @ D
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)

    return z


def baseline_correct_batch(X: np.ndarray, lam: float = 1e5, p: float = 0.01, niter: int = 10) -> np.ndarray:
    corrected = []
    for row in X:
        base = baseline_als(row, lam=lam, p=p, niter=niter)
        corrected.append(row - base)
    return np.vstack(corrected)


def normalize_batch(X: np.ndarray, method: str = "l2") -> np.ndarray:
    X = X.copy()

    if method == "l2":
        norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
        return X / norms

    elif method == "snv":
        mu = X.mean(axis=1, keepdims=True)
        sigma = X.std(axis=1, keepdims=True) + 1e-12
        return (X - mu) / sigma

    elif method == "auc":
        area = np.trapz(np.abs(X), axis=1).reshape(-1, 1) + 1e-12
        return X / area

    else:
        raise ValueError(f"Неизвестный метод нормализации: {method}")


def preprocess_batch(
    wave: np.ndarray,
    X: np.ndarray,
    crop_range: Tuple[float, float] = (700, 1800),
    despike_threshold: float = 5.0,
    smooth_window: int = 9,
    smooth_poly: int = 3,
    baseline_lam: float = 1e5,
    baseline_p: float = 0.01,
    norm_method: str = "l2",
):
    wave2, X2 = crop_spectra(wave, X, crop_range[0], crop_range[1])
    X2 = despike_batch(X2, threshold=despike_threshold, window=3)
    X2 = smooth_batch(X2, window_length=smooth_window, polyorder=smooth_poly)
    X2 = baseline_correct_batch(X2, lam=baseline_lam, p=baseline_p, niter=10)
    X2 = normalize_batch(X2, method=norm_method)
    return wave2, X2


# =========================================================
# Загрузка датасета
# =========================================================

def load_dataset(root_dir: Path) -> Tuple[pd.DataFrame, Dict[str, dict]]:
    """
    Возвращает:
    - metadata_df: по каждому файлу
    - raw_data_dict[file_id] = {
        meta,
        wave,
        X_pixels,     # (n_pixels, n_wave)
        X_agg         # (n_wave,)
    }
    """
    metas = discover_files(root_dir)
    if not metas:
        raise RuntimeError("Файлы .txt не найдены")

    rows = []
    raw_data = {}

    for i, meta in enumerate(metas):
        print(f"[{i+1}/{len(metas)}] Reading: {meta.filepath}")
        df = robust_read_txt(meta.filepath)
        x_unique, y_unique, wave, cube = long_to_cube(df)
        X_pixels = flatten_cube(cube)

        # удаляем полностью пустые/NaN-пиксели
        valid_mask = ~np.isnan(X_pixels).all(axis=1)
        X_pixels = X_pixels[valid_mask]

        X_agg = aggregate_spectra(X_pixels, method="median")

        file_id = f"{meta.label_name}__{meta.mouse_id}__{meta.sample_id}"

        rows.append({
            "file_id": file_id,
            "filepath": meta.filepath,
            "label_name": meta.label_name,
            "label": meta.label,
            "mouse_id": meta.mouse_id,
            "sample_id": meta.sample_id,
            "region": meta.region,
            "n_x": len(x_unique),
            "n_y": len(y_unique),
            "n_pixels_valid": len(X_pixels),
            "n_wave": len(wave),
            "wave_min": float(np.min(wave)),
            "wave_max": float(np.max(wave)),
        })

        raw_data[file_id] = {
            "meta": meta,
            "wave": wave,
            "X_pixels": X_pixels,
            "X_agg": X_agg,
        }

    metadata_df = pd.DataFrame(rows)
    return metadata_df, raw_data


# =========================================================
# EDA
# =========================================================

def eda_summary(metadata_df: pd.DataFrame):
    print("\n=== DATASET SUMMARY ===")
    print(metadata_df.head())
    print("\nFiles per class:")
    print(metadata_df["label_name"].value_counts())

    print("\nFiles per mouse_id:")
    print(metadata_df.groupby(["label_name", "mouse_id"]).size())

    print("\nRegions:")
    print(metadata_df["region"].value_counts())

    print("\nPixels stats:")
    print(metadata_df["n_pixels_valid"].describe())


def check_wave_consistency(raw_data: Dict[str, dict], tol: float = 1e-8) -> bool:
    waves = [v["wave"] for v in raw_data.values()]
    ref = waves[0]

    for i, w in enumerate(waves[1:], start=1):
        if len(w) != len(ref):
            print(f"[WARN] Wave length mismatch at file index={i}")
            return False
        if not np.allclose(w, ref, atol=tol, rtol=0):
            print(f"[WARN] Wave values mismatch at file index={i}")
            return False

    print("[OK] All wave axes are consistent.")
    return True


def interpolate_to_reference(raw_data: Dict[str, dict], reference_wave: np.ndarray):
    """
    Если оси #Wave различаются — интерполируем все спектры на reference_wave.
    """
    for file_id, item in raw_data.items():
        wave = item["wave"]
        if len(wave) == len(reference_wave) and np.allclose(wave, reference_wave):
            continue

        Xp = item["X_pixels"]
        new_X = []
        for row in Xp:
            f = interp1d(wave, row, kind="linear", bounds_error=False, fill_value="extrapolate")
            new_X.append(f(reference_wave))
        new_X = np.vstack(new_X)

        item["wave"] = reference_wave.copy()
        item["X_pixels"] = new_X
        item["X_agg"] = aggregate_spectra(new_X, method="median")


def plot_random_raw_vs_processed(raw_data: Dict[str, dict], n_files: int = 3):
    file_ids = list(raw_data.keys())[:n_files]
    fig, axes = plt.subplots(n_files, 2, figsize=(12, 4 * n_files))
    if n_files == 1:
        axes = np.array([axes])

    for i, file_id in enumerate(file_ids):
        item = raw_data[file_id]
        wave = item["wave"]
        X = item["X_pixels"]

        raw_spec = X[np.random.randint(0, len(X))]
        wave_p, X_p = preprocess_batch(wave, raw_spec[None, :])
        proc_spec = X_p[0]

        axes[i, 0].plot(wave, raw_spec)
        axes[i, 0].set_title(f"RAW: {file_id}")
        axes[i, 0].set_xlabel("Wave")
        axes[i, 0].set_ylabel("Intensity")

        axes[i, 1].plot(wave_p, proc_spec)
        axes[i, 1].set_title(f"PROCESSED: {file_id}")
        axes[i, 1].set_xlabel("Wave")
        axes[i, 1].set_ylabel("Intensity")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "raw_vs_processed_examples.png", dpi=200)
    plt.close()


def plot_class_mean_spectra(raw_data: Dict[str, dict], use_processed: bool = True):
    class_specs = {0: [], 1: [], 2: []}
    common_wave = None

    for item in raw_data.values():
        wave = item["wave"]
        X = item["X_pixels"]
        label = item["meta"].label

        if use_processed:
            wave, X = preprocess_batch(wave, X)

        agg = aggregate_spectra(X, method="median")
        class_specs[label].append(agg)

        if common_wave is None:
            common_wave = wave

    plt.figure(figsize=(12, 6))
    for label, arrs in class_specs.items():
        if len(arrs) == 0:
            continue
        mean_spec = np.mean(np.vstack(arrs), axis=0)
        std_spec = np.std(np.vstack(arrs), axis=0)

        plt.plot(common_wave, mean_spec, label=INV_CLASS_MAP[label])
        plt.fill_between(common_wave, mean_spec - std_spec, mean_spec + std_spec, alpha=0.2)

    plt.title("Mean spectra by class")
    plt.xlabel("Wave (cm^-1)")
    plt.ylabel("Normalized intensity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "mean_spectra_by_class.png", dpi=200)
    plt.close()


# =========================================================
# Формирование выборки
# =========================================================

def build_samplewise_dataset(
    raw_data: Dict[str, dict],
    agg_method: str = "median",
    use_processed: bool = True,
):
    """
    Один объект = один файл.
    """
    X_list, y_list, groups, file_ids = [], [], [], []
    wave_ref = None

    for file_id, item in raw_data.items():
        wave = item["wave"]
        X = item["X_pixels"]

        if use_processed:
            wave, X = preprocess_batch(wave, X)

        X_agg = aggregate_spectra(X, method=agg_method)

        X_list.append(X_agg)
        y_list.append(item["meta"].label)
        groups.append(item["meta"].mouse_id)
        file_ids.append(file_id)

        if wave_ref is None:
            wave_ref = wave

    X = np.vstack(X_list)
    y = np.array(y_list)
    groups = np.array(groups)

    return wave_ref, X, y, groups, file_ids


def build_pixelwise_dataset(
    raw_data: Dict[str, dict],
    max_pixels_per_file: Optional[int] = 500,
    use_processed: bool = True,
):
    """
    Один объект = один спектр пикселя.
    Для борьбы с дисбалансом можно ограничивать число спектров с файла.
    """
    X_list, y_list, groups, parent_files = [], [], [], []
    wave_ref = None

    for file_id, item in raw_data.items():
        wave = item["wave"]
        X = item["X_pixels"]

        if use_processed:
            wave, X = preprocess_batch(wave, X)

        if max_pixels_per_file is not None and len(X) > max_pixels_per_file:
            idx = np.random.choice(len(X), size=max_pixels_per_file, replace=False)
            X = X[idx]

        X_list.append(X)
        y_list.append(np.full(len(X), item["meta"].label))
        groups.append(np.array([item["meta"].mouse_id] * len(X)))
        parent_files.extend([file_id] * len(X))

        if wave_ref is None:
            wave_ref = wave

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    groups = np.concatenate(groups)
    parent_files = np.array(parent_files)

    return wave_ref, X, y, groups, parent_files


# =========================================================
# Модели
# =========================================================

def make_baseline_models():
    models = {
        "pca_svm": Pipeline([
            ("pca", PCA(n_components=0.99, svd_solver="full")),
            ("clf", SVC(C=3.0, kernel="rbf", gamma="scale", probability=True, class_weight="balanced", random_state=42))
        ]),
        "rf": RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        ),
    }
    return models


def evaluate_group_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    models: Dict[str, object],
    n_splits: int = 5,
    dataset_name: str = "samplewise",
):
    unique_groups = np.unique(groups)
    n_splits = min(n_splits, len(unique_groups))
    if n_splits < 2:
        raise ValueError("Недостаточно уникальных групп для GroupKFold.")

    gkf = GroupKFold(n_splits=n_splits)
    results = []

    for model_name, model in models.items():
        fold_reports = []
        y_true_all = []
        y_pred_all = []

        print(f"\n=== MODEL: {model_name} | DATASET: {dataset_name} ===")

        for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            model.fit(X_tr, y_tr)
            pred = model.predict(X_va)

            acc = accuracy_score(y_va, pred)
            bacc = balanced_accuracy_score(y_va, pred)
            macro_f1 = f1_score(y_va, pred, average="macro")

            print(f"Fold {fold}: acc={acc:.4f} | bacc={bacc:.4f} | macro_f1={macro_f1:.4f}")

            fold_reports.append({
                "fold": fold,
                "acc": acc,
                "bacc": bacc,
                "macro_f1": macro_f1,
            })
            y_true_all.extend(y_va.tolist())
            y_pred_all.extend(pred.tolist())

        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)

        final_report = {
            "dataset": dataset_name,
            "model": model_name,
            "acc_mean": float(np.mean([x["acc"] for x in fold_reports])),
            "acc_std": float(np.std([x["acc"] for x in fold_reports])),
            "bacc_mean": float(np.mean([x["bacc"] for x in fold_reports])),
            "bacc_std": float(np.std([x["bacc"] for x in fold_reports])),
            "macro_f1_mean": float(np.mean([x["macro_f1"] for x in fold_reports])),
            "macro_f1_std": float(np.std([x["macro_f1"] for x in fold_reports])),
            "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist(),
            "classification_report": classification_report(
                y_true_all, y_pred_all,
                target_names=[INV_CLASS_MAP[i] for i in sorted(INV_CLASS_MAP.keys())],
                output_dict=True,
                zero_division=0,
            ),
        }
        results.append(final_report)

    results_df = pd.DataFrame([{
        "dataset": r["dataset"],
        "model": r["model"],
        "acc_mean": r["acc_mean"],
        "acc_std": r["acc_std"],
        "bacc_mean": r["bacc_mean"],
        "bacc_std": r["bacc_std"],
        "macro_f1_mean": r["macro_f1_mean"],
        "macro_f1_std": r["macro_f1_std"],
    } for r in results])

    return results, results_df


# =========================================================
# Интерпретация признаков
# =========================================================

def fit_rf_and_feature_importance(X: np.ndarray, y: np.ndarray, wave: np.ndarray, top_k: int = 30):
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    )
    rf.fit(X, y)
    imp = rf.feature_importances_

    feat_df = pd.DataFrame({
        "wave": wave,
        "importance": imp
    }).sort_values("importance", ascending=False)

    plt.figure(figsize=(12, 5))
    plt.plot(wave, imp)
    plt.title("RandomForest feature importance over wave axis")
    plt.xlabel("Wave (cm^-1)")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "rf_feature_importance.png", dpi=200)
    plt.close()

    feat_df.head(top_k).to_csv(OUTPUT_DIR / "top_rf_features.csv", index=False)
    return feat_df


# =========================================================
# Основной сценарий
# =========================================================

def main():
    # 1) загрузка
    metadata_df, raw_data = load_dataset(ROOT_DIR)
    metadata_df.to_csv(OUTPUT_DIR / "metadata_summary.csv", index=False)

    # 2) EDA summary
    eda_summary(metadata_df)

    # 3) Проверка一致ности wave
    wave_ok = check_wave_consistency(raw_data)
    if not wave_ok:
        # создаем reference wave как ось первого файла
        first_key = list(raw_data.keys())[0]
        ref_wave = raw_data[first_key]["wave"]
        print("[INFO] Interpolating all spectra to reference wave axis...")
        interpolate_to_reference(raw_data, ref_wave)
        check_wave_consistency(raw_data)

    # 4) Визуальные sanity checks
    plot_random_raw_vs_processed(raw_data, n_files=min(3, len(raw_data)))
    plot_class_mean_spectra(raw_data, use_processed=True)

    # 5) Sample-wise dataset
    wave_sw, X_sw, y_sw, groups_sw, file_ids_sw = build_samplewise_dataset(
        raw_data,
        agg_method="median",
        use_processed=True
    )

    # 6) Pixel-wise dataset
    wave_pw, X_pw, y_pw, groups_pw, parent_files_pw = build_pixelwise_dataset(
        raw_data,
        max_pixels_per_file=300,   # уменьшайте/увеличивайте под RAM
        use_processed=True
    )

    # 7) Baseline models
    models = make_baseline_models()

    # 8) Group CV: sample-wise
    sw_results, sw_results_df = evaluate_group_cv(
        X_sw, y_sw, groups_sw, models,
        n_splits=5,
        dataset_name="samplewise"
    )
    sw_results_df.to_csv(OUTPUT_DIR / "cv_results_samplewise.csv", index=False)

    # 9) Group CV: pixel-wise
    pw_results, pw_results_df = evaluate_group_cv(
        X_pw, y_pw, groups_pw, models,
        n_splits=5,
        dataset_name="pixelwise"
    )
    pw_results_df.to_csv(OUTPUT_DIR / "cv_results_pixelwise.csv", index=False)

    # 10) Интерпретация: importance на sample-wise
    feat_df = fit_rf_and_feature_importance(X_sw, y_sw, wave_sw, top_k=30)

    # 11) Итоговый json-репорт
    final_report = {
        "n_files": int(len(metadata_df)),
        "n_unique_mice": int(metadata_df["mouse_id"].nunique()),
        "class_distribution": metadata_df["label_name"].value_counts().to_dict(),
        "samplewise_results": sw_results,
        "pixelwise_results": pw_results,
        "top_rf_features": feat_df.head(20).to_dict(orient="records"),
    }

    with open(OUTPUT_DIR / "final_report.json", "w", encoding="utf-8") as f:
        json.dump(final_report, f, ensure_ascii=False, indent=2)

    print("\n[DONE] Results saved to:", OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()

[1/237] Reading: E:\Nuclear_Hack_2026\data\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_1.txt
[2/237] Reading: E:\Nuclear_Hack_2026\data\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_2.txt
[3/237] Reading: E:\Nuclear_Hack_2026\data\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place5_1.txt
[4/237] Reading: E:\Nuclear_Hack_2026\data\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place5_2.txt
[5/237] Reading: E:\Nuclear_Hack_2026\data\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place6_1.txt
[6/237] Reading: E:\Nuclear_Hack_2026\data\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place6_2.txt
[7/237] Reading: E:\Nuclear_Hack_2026\data\control\mk1\cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x

In [2]:
filepath = r'E:\Nuclear_Hack_2026\data'